# Firm Production and Agglomeration

> Production functions, agglomeration economies, and wage determination


In [1]:
# | default_exp production

In [2]:
# | export
import numpy as np

## Why Are Big Cities More Productive?

A key empirical fact: **larger cities have higher productivity and wages**.

**Evidence**:

- Doubling city size increases wages by ~3-8%
- This is known as the **urban wage premium** or **agglomeration benefits**

**Why does this happen?** Three main mechanisms (Marshall's trinity):

1. **Knowledge spillovers**: Learn from nearby workers and firms
2. **Labour market matching**: Find better job/worker matches in thick markets
3. **Input sharing**: Access to specialized suppliers and services

## The Production Function

Firms produce output using labour:

$$Y = A \cdot L^{\alpha}$$

where:

- $Y$ = output
- $A$ = total factor productivity (TFP)
- $L$ = labour (number of workers)
- $\alpha$ = labour share (typically 0.6-0.7)

**Diminishing returns**: As $L$ increases, output grows, but at a decreasing rate (since $\alpha < 1$).

## Agglomeration Economies

The key innovation: **productivity increases with city size**:

$$A = \bar{A} \cdot L^{\eta}$$

where:

- $\bar{A}$ = base TFP
- $\eta$ = **agglomeration elasticity** (typically 0.02-0.08)

This means larger cities have higher TFP!

### Agglomeration Elasticity

The parameter $\eta$ measures how much productivity increases with city size:

$$\eta = \frac{\% \Delta A}{\% \Delta L}$$

**Typical estimates**:

- Academic consensus: $\eta \approx 0.02 - 0.08$ (2-8% TFP gain from doubling city size)
- Hsieh-Moretti (2019): $\eta = 0.25$ (very strong agglomeration!)
- Cooped Up (2024): $\eta = 0.25$ (follows Hsieh-Moretti)

## Combining Production and Agglomeration

Putting it together:

$$Y = \bar{A} \cdot L^{\alpha} \cdot L^{\eta} = \bar{A} \cdot L^{\alpha + \eta}$$

**Key insight**: The effective elasticity of output w.r.t. labour is $\alpha + \eta > \alpha$.

Agglomeration creates **increasing returns to scale** at the city level!


In [ ]:
# | export
def calculate_tfp(
    population: float,  # City population (= labour)
    base_tfp: float,  # Base TFP (productivity when L=1)
    eta: float,  # Agglomeration elasticity
) -> float:
    """Calculate total factor productivity with agglomeration."""
    return base_tfp * (population**eta)


def calculate_output(
    population: float,  # City population (= labour)
    base_tfp: float,  # Base TFP
    alpha: float = 0.65,  # Labour share in production
    eta: float = 0.04,  # Agglomeration elasticity
) -> float:
    """Calculate total output with agglomeration economies."""
    tfp = calculate_tfp(population, base_tfp, eta)
    return tfp * (population**alpha)


def calculate_wage(
    population: float,  # City population (= labour)
    base_tfp: float,  # Base TFP
    alpha: float = 0.65,  # Labour share in production
    eta: float = 0.04,  # Agglomeration elasticity
) -> float:
    """Calculate equilibrium wage as marginal product of labour."""
    tfp = calculate_tfp(population, base_tfp, eta)
    return alpha * tfp * (population ** (alpha - 1))


def output_per_worker(
    population: float,  # City population
    base_tfp: float,  # Base TFP
    alpha: float = 0.65,  # Labour share
    eta: float = 0.04,  # Agglomeration elasticity
) -> float:
    """Calculate output per worker (labour productivity)."""
    return calculate_output(population, base_tfp, alpha, eta) / population


def agglomeration_factor(
    population: float,  # Population of city of interest
    reference_population: float,  # Population of reference city
    eta: float,  # Agglomeration elasticity
) -> float:
    """Calculate the TFP multiplier from agglomeration relative to a reference city."""
    return (population / reference_population) ** eta

Examples:

In [ ]:
# Calculate TFP with agglomeration
tfp = calculate_tfp(population=1_000_000, base_tfp=100, eta=0.04)
assert tfp > 100  # Should be higher due to agglomeration
assert np.isclose(tfp, 100 * (1_000_000**0.04))

# Test that larger cities have higher TFP
tfp_small = calculate_tfp(100_000, 100, 0.04)
tfp_large = calculate_tfp(1_000_000, 100, 0.04)
assert tfp_large > tfp_small

# Test wage = alpha * output per worker
wage = calculate_wage(1_000_000, 100, 0.65, 0.04)
output_pw = output_per_worker(1_000_000, 100, 0.65, 0.04)
assert np.isclose(wage, 0.65 * output_pw, rtol=0.01)

## Demonstrating Agglomeration Effects


In [4]:
# Compare cities of different sizes
base_tfp = 100
alpha = 0.65
eta = 0.04

populations = [100_000, 500_000, 1_000_000, 5_000_000, 10_000_000]

print("Effect of City Size on Productivity and Wages")
print("=" * 80)
print(f"{'Population':>12} {'TFP':>10} {'Output':>15} {'Wage':>10} {'Output/Worker':>15}")
print("-" * 80)

for pop in populations:
    tfp = calculate_tfp(pop, base_tfp, eta)
    output = calculate_output(pop, base_tfp, alpha, eta)
    wage = calculate_wage(pop, base_tfp, alpha, eta)
    output_pw = output_per_worker(pop, base_tfp, alpha, eta)

    print(f"{pop:>12,} {tfp:>10.1f} {output:>15,.0f} {wage:>10.2f} {output_pw:>15.2f}")

print("-" * 80)
print("\nKey insight: Larger cities have higher TFP, output, wages, and productivity!")

Effect of City Size on Productivity and Wages
  Population        TFP          Output       Wage   Output/Worker
--------------------------------------------------------------------------------
     100,000      158.5         281,838       1.83            2.82
     500,000      169.0         855,637       1.11            1.71
   1,000,000      173.8       1,380,384       0.90            1.38
   5,000,000      185.3       4,190,726       0.54            0.84
  10,000,000      190.5       6,760,830       0.44            0.68
--------------------------------------------------------------------------------

Key insight: Larger cities have higher TFP, output, wages, and productivity!


## Comparing Agglomeration Parameters

Let's see how different values of $\eta$ affect the urban wage premium.


In [5]:
# Compare wage premia with different agglomeration elasticities
small_city_pop = 500_000
large_city_pop = 5_000_000  # 10x larger

etas = [
    (0.02, "Conservative (η=0.02)"),
    (0.04, "Standard (η=0.04)"),
    (0.08, "High (η=0.08)"),
    (0.25, "Hsieh-Moretti (η=0.25)"),
]

print(f"Urban Wage Premium: Large City ({large_city_pop:,}) vs Small City ({small_city_pop:,})")
print("=" * 80)
print(f"{'Scenario':<30} {'Small City Wage':>18} {'Large City Wage':>18} {'Wage Premium':>15}")
print("-" * 80)

for eta, label in etas:
    wage_small = calculate_wage(small_city_pop, base_tfp, alpha, eta)
    wage_large = calculate_wage(large_city_pop, base_tfp, alpha, eta)
    premium_pct = (wage_large / wage_small - 1) * 100

    print(f"{label:<30} {wage_small:>18.2f} {wage_large:>18.2f} {premium_pct:>14.1f}%")

print("-" * 80)
print("\nInterpretation: Higher η → stronger agglomeration → larger wage differences")

Urban Wage Premium: Large City (5,000,000) vs Small City (500,000)
Scenario                          Small City Wage    Large City Wage    Wage Premium
--------------------------------------------------------------------------------
Conservative (η=0.02)                        0.86               0.40          -53.2%
Standard (η=0.04)                            1.11               0.54          -51.0%
High (η=0.08)                                1.88               1.01          -46.3%
Hsieh-Moretti (η=0.25)                      17.50              13.90          -20.6%
--------------------------------------------------------------------------------

Interpretation: Higher η → stronger agglomeration → larger wage differences


## The Agglomeration Multiplier

How much more productive is a city of size $L$ compared to a reference city?


In [6]:
# Compare London vs other UK cities
reference_pop = 500_000  # Typical mid-size UK city
eta = 0.04

uk_cities = [
    ("Mid-size city (reference)", 500_000),
    ("Birmingham", 1_100_000),
    ("Manchester", 2_800_000),
    ("London", 9_000_000),
]

print("TFP Multiplier from Agglomeration (relative to mid-size city)")
print("=" * 60)
print(f"{'City':<30} {'Population':>15} {'TFP Multiplier':>15}")
print("-" * 60)

for city_name, pop in uk_cities:
    multiplier = agglomeration_factor(pop, reference_pop, eta)
    print(f"{city_name:<30} {pop:>15,} {multiplier:>15.2f}x")

print("-" * 60)
print(f"\nWith η={eta}, London is {agglomeration_factor(9_000_000, 500_000, eta):.2f}x more productive")
print("than a mid-size city purely from agglomeration effects!")

TFP Multiplier from Agglomeration (relative to mid-size city)
City                                Population  TFP Multiplier
------------------------------------------------------------
Mid-size city (reference)              500,000            1.00x
Birmingham                           1,100,000            1.03x
Manchester                           2,800,000            1.07x
London                               9,000,000            1.12x
------------------------------------------------------------

With η=0.04, London is 1.12x more productive
than a mid-size city purely from agglomeration effects!


## Tests


In [7]:
# | hide
# Test TFP calculation
tfp = calculate_tfp(1_000_000, 100, 0.04)
assert tfp > 100  # Should be higher due to agglomeration
assert np.isclose(tfp, 100 * (1_000_000**0.04))

# Test that larger cities have higher TFP
tfp_small = calculate_tfp(100_000, 100, 0.04)
tfp_large = calculate_tfp(1_000_000, 100, 0.04)
assert tfp_large > tfp_small

# Test output calculation
output = calculate_output(1_000_000, 100, 0.65, 0.04)
assert output > 0

# Test wage calculation
wage = calculate_wage(1_000_000, 100, 0.65, 0.04)
assert wage > 0

# Test that wage = alpha * output per worker
output_pw = output_per_worker(1_000_000, 100, 0.65, 0.04)
assert np.isclose(wage, 0.65 * output_pw, rtol=0.01)

# Test agglomeration factor
factor = agglomeration_factor(1_000_000, 100_000, 0.04)
assert factor > 1.0  # Larger city should be more productive
assert np.isclose(factor, 10**0.04)

# Test wage ratio formula consistency
# Note: With standard parameters (alpha=0.65, eta=0.04), alpha + eta < 1
# so wages decrease with city size holding base_tfp constant.
# The urban wage premium comes from cities with different base_tfp in equilibrium.
wage_1M = calculate_wage(1_000_000, 100, 0.65, 0.04)
wage_2M = calculate_wage(2_000_000, 100, 0.65, 0.04)
expected_ratio = 2 ** (0.65 + 0.04 - 1)  # Should equal wage_2M / wage_1M
assert np.isclose(wage_2M / wage_1M, expected_ratio, rtol=0.01)

## Why This Matters

Agglomeration economies create a powerful force pulling workers toward large cities:

1. **Big cities are more productive** → firms can pay higher wages
2. **Workers want higher wages** → they migrate to big cities
3. **But rents rise** (especially with inelastic housing supply)
4. **Equilibrium** balances productivity gains against rent increases

### Policy Implications

If housing supply is constrained in productive cities:

- Rents spike
- Workers are pushed out
- They end up in less productive locations
- **National GDP falls**

The strength of agglomeration ($\eta$) determines **how much GDP is lost**:

- Higher $\eta$ → stronger agglomeration → bigger losses from constraints
- Hsieh-Moretti use $\eta = 0.25$ and find 3.7-8.9% GDP loss
- With lower $\eta \approx 0.04$, the loss would be smaller but still significant


## Key Insights

1. **Agglomeration economies**: Larger cities are more productive (η typically 0.02-0.08)
2. **Wage premium**: Big cities pay higher wages due to higher productivity
3. **Increasing returns**: At the city level, production has increasing returns (α + η > α)
4. **Debate over η**: Academic estimates vary from 0.02 to 0.25, which matters hugely for policy
5. **Spatial misallocation**: If productive cities can't accommodate workers, national output suffers


In [8]:
# | hide
import nbdev

nbdev.nbdev_export()